# TSTR (Train Synthetic, Test Real)

# 1. Installing libraries

In [ ]:
!pip install -q gdown

In [ ]:
!pip install uv

In [ ]:
!uv pip install torch pytorch_lightning xgboost optuna lightgbm catboost tqdm

# 2. Importing libraries and taking configurations

In [ ]:
import os
import json
import warnings
import time
import resource
from datetime import datetime

import numpy as np
import pandas as pd

import gdown
import optuna
from google.colab import drive

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import torch
import pytorch_lightning as pl

from tqdm import tqdm

Ensuring all scikit learn algorithms run with GPUs

In [ ]:
%load_ext cuml.accel

Connecting to google drive

In [ ]:
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/TSTR_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42)

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__} | PL {pl.__version__} | {DEVICE}")

SEEDS = [42, 137, 256, 1024, 7, 314, 999, 2048, 55, 8080,
         101, 202, 303, 404, 505, 606, 707, 808, 909, 1111]

N_TRIALS = 40

In [ ]:
_WALL_CLOCK = {}   # "model__dataset__mode" → seconds
_PEAK_MEM   = {}   # same key → ΔRSS in MB

def _rss_mb():
    """Current RSS in MB (Linux)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

def _snap(label, t0, m0):
    _WALL_CLOCK[label] = time.time() - t0
    _PEAK_MEM[label]   = _rss_mb() - m0

#3. Importing wells data

Real

Zone 1:
https://docs.google.com/spreadsheets/d/1opPeaM4t-_co9wDNSiK8zilHbH0xjDRHP8PkS1ZrgoM/edit?gid=1510789613#gid=1510789613

https://docs.google.com/spreadsheets/d/1jaR5XHmUoo6fYYw6E3sCl7lVuvXIbKZ1HwYssAxlFyE/edit?gid=1701454767#gid=1701454767

https://docs.google.com/spreadsheets/d/1xJKh8kgPX_WqfhOZ0fabv9VnlBcWMgKJoIpAo_GVYAA/edit?gid=1968934774#gid=1968934774


Zone 2:

https://docs.google.com/spreadsheets/d/19iGURry9WwIM4F8zr2KbRHPdxNhnQuYFEJn8bZLaScU/edit?gid=1217895151#gid=1217895151


https://docs.google.com/spreadsheets/d/1h6za5W9a-plukPQiJko_dhLs1td0i5ZHjLf_XeIRusQ/edit?gid=578251623#gid=578251623

https://docs.google.com/spreadsheets/d/1cyRbeBMS8DI61yJJrlAt7NYI7mcwzWc_lwlOdvRuWYs/edit?gid=1626524731#gid=1626524731



Zone 3:

https://docs.google.com/spreadsheets/d/1rf8HPssvBa49fybUmaYwP19PSIvUJwWEm3V_nsGPUlk/edit?gid=845988251#gid=845988251

https://docs.google.com/spreadsheets/d/11IZfbGKVNOEsk7ONL0Xj_VPPECqlB6ScJcEQ41Wy7Hw/edit?gid=1646218926#gid=1646218926

https://docs.google.com/spreadsheets/d/1Xzy4w-ju2Pq06CZHbrWHNW1gK-VS7PYavcm9AccjLAQ/edit?gid=579143726#gid=579143726


Synthetic

Zone 1:

https://docs.google.com/spreadsheets/d/1qq2oL0fWNKM-zzZVxukdg_vswa-f0Un5r1U9_20YJNE/edit?gid=1297109795#gid=1297109795

https://docs.google.com/spreadsheets/d/1KjoO_y0xkn02kOx6oE0IFu-wHCc1qqRztpXvsa0UQc8/edit?gid=1231346322#gid=1231346322

https://docs.google.com/spreadsheets/d/1PyfLTVchbW7IOn6i_rjLXKprrLLGScCtVp1ZBQNhtzI/edit?gid=1144871303#gid=1144871303

Zone 2:

https://docs.google.com/spreadsheets/d/1sjyFegD384N-aSGsnMezEB14-Hc16V0cM-r4FLiLl4I/edit?gid=387103156#gid=387103156

https://docs.google.com/spreadsheets/d/1dMsJDYi43GyPP6vWejXYf-Hi55q-b6K7QHlJxHFW4aw/edit?gid=1470285094#gid=1470285094

https://docs.google.com/spreadsheets/d/1gEeZ0j6kWyEjuDoW0xcDLeeV2uzIPvyLuF9NxEiixO0/edit?gid=1313257235#gid=1313257235


Zone 3:


https://docs.google.com/spreadsheets/d/1EdDO6n4D9ghml_gdgByFaFE9p4KFYqnG5XS8HhaMEO0/edit?gid=634135141#gid=634135141


https://docs.google.com/spreadsheets/d/1p7uRWawNAwGjnMMMvt2PhHj-JAPoWY5lzZGQnl3eghQ/edit?gid=622018849#gid=622018849


https://docs.google.com/spreadsheets/d/13ojON4UnZ5tbenx3mamepzF22O4U6sIkqmaC9g28xEY/edit?gid=687284441#gid=687284441

In [ ]:
import gdown
from tqdm import tqdm

file_ids = [
    # Synthetic Zone 1
    '1HfCNjR1Vy4rGwE9Lq8ptk2x5EdiaDCZOJYZaXEU2HIc',
    '1h1DkI8kspgH7KaclEDpAR2-vbXvwQPxIx_OWfxhXc-Q',
    '1-jeIIedx2nkRhd3vleuR3X1bxlwWnhsQMUbzhzPaB4E',
    # Synthetic Zone 2
    '1d7KCQYMwCM4vsbv9I2SbChwzRJafLLwjW8S5qto6beU',
    '1Y4evekxe84WU2Xp5teaKnWFKkBAIwGw2usdOb8RvJqc',
    '1_4aSHTf-BurZdB2RhpyM1CoSQj0CLmJs836gW89uoII',
    # Synthetic Zone 3
    '1MQeVpLkz_nMjWvD7I-F9d4XecAdmVlaZp31YsBEkagY',
    '1vTo0gehq6mjyFvKTg0ufwXyfwPb2lCHiUteS8WPMzEk',
    '1il51wkQt0BaqPByiHPKturGRhELzL4qeA9zC81UoWUo',
    # Real Zone 1
    '1opPeaM4t-_co9wDNSiK8zilHbH0xjDRHP8PkS1ZrgoM',
    '1jaR5XHmUoo6fYYw6E3sCl7lVuvXIbKZ1HwYssAxlFyE',
    '1xJKh8kgPX_WqfhOZ0fabv9VnlBcWMgKJoIpAo_GVYAA',
    # Real Zone 2
    '19iGURry9WwIM4F8zr2KbRHPdxNhnQuYFEJn8bZLaScU',
    '1h6za5W9a-plukPQiJko_dhLs1td0i5ZHjLf_XeIRusQ',
    '1cyRbeBMS8DI61yJJrlAt7NYI7mcwzWc_lwlOdvRuWYs',
    # Real Zone 3
    '1rf8HPssvBa49fybUmaYwP19PSIvUJwWEm3V_nsGPUlk',
    '11IZfbGKVNOEsk7ONL0Xj_VPPECqlB6ScJcEQ41Wy7Hw',
    '1Xzy4w-ju2Pq06CZHbrWHNW1gK-VS7PYavcm9AccjLAQ',
]

for file_id in tqdm(file_ids, desc="Downloading Files"):
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, quiet=False)

In [ ]:
import pandas as pd


file_paths = {
    "path_1_real": "/content/real_MISSA-KESWAL-01.xlsx",
    "path_1_synth": "/content/synth_MISSA-KESWAL-01.xlsx",
    "path_2_real": "/content/real_MISSA-KESWAL-02.xlsx",
    "path_2_synth": "/content/synth_MISSA-KESWAL-02.xlsx",
    "path_3_real": "/content/real_MISSA-KESWAL-03.xlsx",
    "path_3_synth": "/content/synth_MISSA-KESWAL-03.xlsx",
    "path_4_real": "/content/real_PINDORI-1.xlsx",
    "path_4_synth": "/content/synth_PINDORI-1.xlsx",
    "path_5_real": "/content/real_PINDORI-2.xlsx",
    "path_5_synth": "/content/synth_PINDORI-2.xlsx",
    "path_6_real": "/content/real_PINDORI-3.xlsx",
    "path_6_synth": "/content/synth_PINDORI-3.xlsx",
    "path_7_real": "/content/real_JOYAMAIR-4.xlsx",
    "path_7_synth": "/content/synth_JOYAMAIR-4.xlsx",
    "path_8_real": "/content/real_MINWAL-2.xlsx",
    "path_8_synth": "/content/synth_MINWAL-2.xlsx",
    "path_9_real": "/content/real_MINWAL-X-1.xlsx",
    "path_9_synth": "/content/synth_MINWAL-X-1.xlsx"
}



In [ ]:
# 2. Dictionary to store the actual DataFrames
dataframes = {}

print(f"{'Dataset Name':<25} | {'Row Count':<10}")
print("-" * 40)

# 3. Loop through, load, and count
for name, path in file_paths.items():
    try:
        df = pd.read_excel(path)

        # Store it in our dictionary so you can access it later (e.g., dataframes['path_1_real'])
        dataframes[name] = df

        # Print the row count
        print(f"{name:<25} | {len(df):<10}")

    except Exception as e:
        print(f"Error loading {name}: {e}")

In [ ]:
for name, df in dataframes.items():
    print(f"{name:<25} | {df.columns.tolist()}")

1. path_2 | 5627 rows
2. path_7 | 6081 rows
3. path_1 | 7019 rows
4. path_3 | 7542 rows
5. path_9 | 8169 rows
6. path_6 | 9463 rows
7. path_8 | 9878 rows
8. path_4 | 41707 rows
9. path_5 | 41995 rows

#4. Processing the data

In [ ]:
print(dataframes['path_7_real'].columns)

In [ ]:
mapping = {
    'RT': 'RES_DEEP',
    'RHOB': 'RHOB_combined',
    'RHOB_COMBINED': 'RHOB_combined',
}

# 1. First, rename the synth columns so they can be matched
synth_keys = [k for k in dataframes.keys() if '_synth' in k]
for key in synth_keys:
    dataframes[key] = dataframes[key].rename(columns=mapping)

# Rename in BOTH synth and real
for k in dataframes:
    dataframes[k] = dataframes[k].rename(columns=mapping)

# 2. Loop through each path number (1-9) to intersect columns
for i in range(1, 10):
    s_key = f'path_{i}_synth'
    r_key = f'path_{i}_real'

    # Check if both keys exist in your dictionary to avoid errors
    if s_key in dataframes and r_key in dataframes:
        # Find columns that exist in BOTH dataframes
        common_cols = dataframes[s_key].columns.intersection(dataframes[r_key].columns)

        # Filter both dataframes to only include these common columns
        dataframes[s_key] = dataframes[s_key][common_cols]
        dataframes[r_key] = dataframes[r_key][common_cols]

In [ ]:
# 3. Create your tuples (now they have identical shapes)
path_1_data = (dataframes['path_1_synth'], dataframes['path_1_real'])
path_2_data = (dataframes['path_2_synth'], dataframes['path_2_real'])
path_3_data = (dataframes['path_3_synth'], dataframes['path_3_real'])
path_4_data = (dataframes['path_4_synth'], dataframes['path_4_real'])
path_5_data = (dataframes['path_5_synth'], dataframes['path_5_real'])
path_6_data = (dataframes['path_6_synth'], dataframes['path_6_real'])
path_7_data = (dataframes['path_7_synth'], dataframes['path_7_real'])
path_8_data = (dataframes['path_8_synth'], dataframes['path_8_real'])
path_9_data = (dataframes['path_9_synth'], dataframes['path_9_real'])


PHYSICAL_BOUNDS = {
    "GR": (0, 200), "DT": (30, 180), "RHOB_combined": (1.2, 2.9),
    "RES_DEEP": (0.01, 10_000), "HP": (500, 15_000), "OB": (2_000, 40_000),
    "DT_NCT": (30, 180), "PPP": (0, 30_000),
}

SENTINEL_VALUES = {-999, -999.25, -999.0}


def clean_well_data(df, outlier_std=5, verbose=True, label=""):
    """
    Clean a single well DataFrame (synthetic OR real).
    1. Drop SPHI column if present.
    2. Replace sentinel values (-999, -999.25) with NaN.
    3. Replace values outside physical bounds with NaN.
    4. Replace extreme outliers (> outlier_std σ) with NaN.
    5. Drop rows with any remaining NaN in log columns.
    """
    df = df.copy()
    n_before = len(df)

    if "SPHI" in df.columns:
        df.drop(columns=["SPHI"], inplace=True)

    log_cols = [c for c in df.columns if c != "DEPTH"]

    for col in log_cols:
        df.loc[df[col].isin(SENTINEL_VALUES), col] = np.nan

    for col in log_cols:
        if col in PHYSICAL_BOUNDS:
            lo, hi = PHYSICAL_BOUNDS[col]
            df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

    for col in log_cols:
        clean_vals = df[col].dropna()
        if len(clean_vals) < 10:
            continue
        mu, sig = clean_vals.mean(), clean_vals.std()
        if sig == 0:
            continue
        df.loc[(df[col] - mu).abs() > outlier_std * sig, col] = np.nan

    df.dropna(subset=log_cols, how='any', inplace=True)
    df.reset_index(drop=True, inplace=True)

    if verbose:
        removed = n_before - len(df)
        pct = 100 * removed / n_before if n_before else 0
        print(f"  [{label}]  {n_before} → {len(df)} rows  "
              f"(removed {removed}, {pct:.1f}%)")
    return df


def clean_all_wells(real_dfs, well_names, outlier_std=5):
    cleaned = []
    for name, df in zip(well_names, real_dfs):
        cleaned.append(clean_well_data(df, outlier_std=outlier_std,
                                       label=f"{name} REAL"))
    return cleaned


def clean_synthetic(df_syn, outlier_std=5, label="SYNTH"):
    return clean_well_data(df_syn, outlier_std=outlier_std, label=label)

In [ ]:
_all_path_tuples = {
    "path_1": path_1_data, "path_2": path_2_data, "path_3": path_3_data,
    "path_4": path_4_data, "path_5": path_5_data, "path_6": path_6_data,
    "path_7": path_7_data, "path_8": path_8_data, "path_9": path_9_data,
}

print("\n=== Cleaning all wells ===")
for _name, (_synth_df, _real_df) in _all_path_tuples.items():
    _cleaned_synth = clean_well_data(_synth_df, label=f"{_name} SYNTH")
    _cleaned_real  = clean_well_data(_real_df,  label=f"{_name} REAL")
    _all_path_tuples[_name] = (_cleaned_synth, _cleaned_real)

path_1_data = _all_path_tuples["path_1"]
path_2_data = _all_path_tuples["path_2"]
path_3_data = _all_path_tuples["path_3"]
path_4_data = _all_path_tuples["path_4"]
path_5_data = _all_path_tuples["path_5"]
path_6_data = _all_path_tuples["path_6"]
path_7_data = _all_path_tuples["path_7"]
path_8_data = _all_path_tuples["path_8"]
path_9_data = _all_path_tuples["path_9"]
print("=== Cleaning complete ===\n")

In [ ]:
order_of_dataset_processing = [
    path_2_data,
    path_7_data,
    path_1_data,
    path_3_data,
    path_9_data,
    path_6_data,
    path_8_data,
    path_4_data,
    path_5_data
]

In [ ]:
# Iterate through each tuple in your ordered list
for i, (synth_df, real_df) in enumerate(order_of_dataset_processing):
    print(f"--- Dataset Group {i+1} ---")

    # Print columns for the Synthetic DataFrame
    print(f"Synthetic Columns: {synth_df.columns.tolist()}")

    # Print columns for the Real DataFrame
    print(f"Real Columns:      {real_df.columns.tolist()}")
    print("\n")

In [ ]:
def processing_tuple(tuple_with_data):
    synth_df, real_df = tuple_with_data
    TARGET = 'PPP'
    EXCLUDE = ['DEPTH']
    features = [c for c in synth_df.columns if c != TARGET and c not in EXCLUDE]
    print(features)

    # TSTR: train/val from synthetic, blind test from real
    train_df, val_df = train_test_split(synth_df, test_size=0.3, random_state=42)

    # Combined synthetic (train + val) for final retraining
    combo = pd.concat([train_df, val_df], ignore_index=True)

    # --- Feature scaler (unchanged) ---
    scaler = StandardScaler()
    all_data = pd.concat([synth_df[features], real_df[features]], ignore_index=True)
    scaler.fit(synth_df[features].values)       # only synth

    X_all_s = scaler.transform(combo[features].values)
    X_test_s = scaler.transform(real_df[features].values)
    X_optuna_train = scaler.transform(train_df[features].values)
    X_optuna_val = scaler.transform(val_df[features].values)

    # --- Target scaler (NEW) ---
    target_scaler = StandardScaler()
    all_targets = pd.concat([synth_df[[TARGET]], real_df[[TARGET]]], ignore_index=True)
    target_scaler.fit(synth_df[[TARGET]].values) # only synth

    y_all = target_scaler.transform(combo[[TARGET]].values).ravel()
    y_test = target_scaler.transform(real_df[[TARGET]].values).ravel()
    y_optuna_train = target_scaler.transform(train_df[[TARGET]].values).ravel()
    y_optuna_val = target_scaler.transform(val_df[[TARGET]].values).ravel()

    return {
        'features': features,
        'scaler': scaler,
        'target_scaler': target_scaler,
        'X_all_s': X_all_s,
        'y_all': y_all,
        'X_test_s': X_test_s,
        'y_test': y_test,
        'X_optuna_train': X_optuna_train,
        'y_optuna_train': y_optuna_train,
        'X_optuna_val': X_optuna_val,
        'y_optuna_val': y_optuna_val,
    }

# 5. Unified TSTR with Pure + Semi (5%, 10%, 20%) per path

In [ ]:
RIDGE_SPEC = {
    "name": "Ridge",
    "build": lambda p: Ridge(**p),
    "suggest": lambda t: {
        "alpha": t.suggest_float("alpha", 1e-4, 100.0, log=True),
    },
}

LIGHTGBM_SPEC = {
    "name": "LightGBM",
    "build": lambda p: lgb.LGBMRegressor(
        **p,
        random_state=42,
        verbosity=-1,
        device="gpu"  # ← GPU Enabled
    ),
    "suggest": lambda t: {
        "n_estimators":      t.suggest_int("n_estimators", 100, 2000),
        "learning_rate":     t.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth":         t.suggest_int("max_depth", 3, 12),
        "num_leaves":        t.suggest_int("num_leaves", 16, 256),
        "min_child_samples": t.suggest_int("min_child_samples", 5, 100),
        "subsample":         t.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         t.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        t.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    },
}

XGBOOST_SPEC = {
    "name": "XGBoost",
    "build": lambda p: xgb.XGBRegressor(
        **p,
        random_state=42,
        verbosity=0,
        tree_method="hist", # ← GPU Enabled
        device="cuda"           # ← GPU Enabled
    ),
    "suggest": lambda t: {
        "n_estimators":     t.suggest_int("n_estimators", 100, 2000),
        "learning_rate":    t.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "max_depth":        t.suggest_int("max_depth", 3, 12),
        "min_child_weight": t.suggest_float("min_child_weight", 1, 20),
        "subsample":        t.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":            t.suggest_float("gamma", 1e-8, 5.0, log=True),
        "reg_alpha":        t.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       t.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    },
}

RANDOM_FOREST_SPEC = {
    "name": "RandomForest",
    "build": lambda p: RandomForestRegressor(**p, random_state=42, n_jobs=-1),
    "suggest": lambda t: {
        "n_estimators":      t.suggest_int("n_estimators", 100, 1500),
        "max_depth":         t.suggest_int("max_depth", 4, 30),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf":  t.suggest_int("min_samples_leaf", 1, 20),
        "max_features":      t.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, 1.0]),
    },
}

CATBOOST_SPEC = {
    "name": "CatBoost",
    "build": lambda p: CatBoostRegressor(
        **p,
        random_seed=42,
        verbose=0,
        task_type="GPU" # ← GPU Enabled
    ),
    "suggest": lambda t: {
        "iterations":          t.suggest_int("iterations", 100, 2000),
        "learning_rate":       t.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "depth":               t.suggest_int("depth", 3, 10),
        "l2_leaf_reg":         t.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": t.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":     t.suggest_float("random_strength", 1e-3, 10.0, log=True),
    },
}

KRR_SPEC = {
    "name": "KRR",
    "build": lambda p: KernelRidge(**p),
    "suggest": lambda t: {
        "kernel": t.suggest_categorical("kernel", ["rbf", "linear"]),
        "alpha":  t.suggest_float("alpha", 1e-4, 100.0, log=True),
        "gamma":  t.suggest_float("gamma", 1e-5, 1.0, log=True),
    },
}

In [ ]:
PATHS_SMALLEST_TO_BIGGEST = [
    ("path_2", path_2_data),
    ("path_7", path_7_data),
    ("path_1", path_1_data),
    ("path_3", path_3_data),
    ("path_9", path_9_data),
    ("path_6", path_6_data),
    ("path_8", path_8_data),
    ("path_4", path_4_data),
    ("path_5", path_5_data),
]

In [ ]:
all_reports = []
_t0_all, _m0_all = time.time(), _rss_mb()

Before running the code, making sure the GPUs for ML models will be used

In [ ]:
%reload_ext cuml.accel

## 5.2 Pure TSTR

In [ ]:
ALL_SPECS = [RIDGE_SPEC, KRR_SPEC, LIGHTGBM_SPEC, XGBOOST_SPEC,
             RANDOM_FOREST_SPEC, CATBOOST_SPEC]

In [ ]:
def objective(trial):
    params = _spec["suggest"](trial)
    model  = _spec["build"](params)
    model.fit(_X_tr, _y_tr)
    return mean_squared_error(_y_va, model.predict(_X_va))

In [ ]:
def run_model(spec, data_tuple, dataset_name):
    """
    1. Optuna finds best hyperparams (one time, seed=42 split).
    2. For each seed: re-split synth → train on synth → test on real.
    3. Report mean ± std across all seeds.
    """
    global _spec, _X_tr, _y_tr, _X_va, _y_va

    name = spec["name"]

    # ── Step 1: Find best hyperparams with Optuna (single seed=42 split) ────
    d = processing_tuple(data_tuple)
    ts = d['target_scaler']

    _spec = spec
    _X_tr, _y_tr = d['X_optuna_train'], d['y_optuna_train']
    _X_va, _y_va = d['X_optuna_val'],   d['y_optuna_val']

    print(f"\n{'='*60}")
    print(f"  {name}  —  {dataset_name}")
    print(f"{'='*60}")

    study = optuna.create_study(direction="minimize")
    _t0, _m0 = time.time(), _rss_mb()
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    _optuna_secs = time.time() - _t0
    best_params = study.best_params
    print(f"  Best params: {best_params}")

    # ── Step 2: Re-train + test across all seeds ────────────────────────────
    synth_df, real_df = data_tuple
    TARGET = 'PPP'
    EXCLUDE = ['DEPTH']
    features = [c for c in synth_df.columns if c != TARGET and c not in EXCLUDE]

    seed_r2, seed_rmse, seed_mae = [], [], []
    seed_val_r2, seed_val_rmse, seed_val_mae = [], [], []
    _t0_seeds, _m0_seeds = time.time(), _rss_mb()


    real_sorted = real_df.sort_values("DEPTH").reset_index(drop=True)
    real_test   = real_sorted.iloc[int(0.70 * len(real_sorted)):].reset_index(drop=True)

    for seed in tqdm(SEEDS, desc=f"  {name} | {dataset_name} | seeds"):
        # Fresh split of synthetic data with this seed
        train_s, val_s = train_test_split(synth_df, test_size=0.3, random_state=seed)

        # Scaler fit on this split's synthetic training data only
        feat_scaler = StandardScaler()
        feat_scaler.fit(train_s[features].values)

        tgt_scaler = StandardScaler()
        tgt_scaler.fit(train_s[[TARGET]].values)

        X_train = feat_scaler.transform(train_s[features].values)
        y_train = tgt_scaler.transform(train_s[[TARGET]].values).ravel()

        X_val = feat_scaler.transform(val_s[features].values)

        X_test = feat_scaler.transform(real_test[features].values)

        # Train with best params from Optuna
        model = spec["build"](best_params)
        model.fit(X_train, y_train)

        # ── Validation metrics (synthetic held-out) ────────────────────────
        val_preds = tgt_scaler.inverse_transform(model.predict(X_val).reshape(-1, 1)).ravel()
        val_truth = val_s[TARGET].values
        seed_val_r2.append(r2_score(val_truth, val_preds))
        seed_val_rmse.append(np.sqrt(mean_squared_error(val_truth, val_preds)))
        seed_val_mae.append(mean_absolute_error(val_truth, val_preds))

        # ── Test metrics (real data) ───────────────────────────────────────
        preds = tgt_scaler.inverse_transform(model.predict(X_test).reshape(-1, 1)).ravel()
        truth = real_test[TARGET].values

        r2  = r2_score(truth, preds)
        rmse = np.sqrt(mean_squared_error(truth, preds))
        mae = mean_absolute_error(truth, preds)

        seed_r2.append(r2)
        seed_rmse.append(rmse)
        seed_mae.append(mae)

    # ── Step 3: Report ──────────────────────────────────────────────────────
    seed_r2       = np.array(seed_r2)
    seed_rmse     = np.array(seed_rmse)
    seed_mae      = np.array(seed_mae)
    seed_val_r2   = np.array(seed_val_r2)
    seed_val_rmse = np.array(seed_val_rmse)
    seed_val_mae  = np.array(seed_val_mae)

    print(f"\n  Results across {len(SEEDS)} seeds:")
    print(f"    Val R²  = {seed_val_r2.mean():.4f} ± {seed_val_r2.std():.4f}   "
          f"(min={seed_val_r2.min():.4f}, max={seed_val_r2.max():.4f})")
    print(f"    Val RMSE= {seed_val_rmse.mean():.4f} ± {seed_val_rmse.std():.4f}")
    print(f"    Val MAE = {seed_val_mae.mean():.4f} ± {seed_val_mae.std():.4f}")
    print(f"    Test R² = {seed_r2.mean():.4f} ± {seed_r2.std():.4f}   "
          f"(min={seed_r2.min():.4f}, max={seed_r2.max():.4f})")
    print(f"    Test RMSE = {seed_rmse.mean():.4f} ± {seed_rmse.std():.4f}")
    print(f"    Test MAE  = {seed_mae.mean():.4f} ± {seed_mae.std():.4f}")
    print(f"    ΔR²     = {(seed_r2.mean() - seed_val_r2.mean()):+.4f}  "
          f"(test - val)")

    if seed_r2.max() < 0:
        print(f"    ⚠️  ALL seeds produced negative R² — "
              f"synthetic data may not transfer for this well.")
    elif seed_r2.mean() < 0:
        pct_pos = 100 * (seed_r2 > 0).sum() / len(seed_r2)
        print(f"    ⚠️  Mean R² is negative but {pct_pos:.0f}% of seeds were positive.")

    # Save checkpoint (best seed)
    best_idx = np.argmax(seed_r2)
    tag = f"{name}_{dataset_name}"

    _seeds_secs = time.time() - _t0_seeds
    _total_secs = _optuna_secs + _seeds_secs
    _label = f"{name}__{dataset_name}__pure"
    _snap(_label, _t0_seeds - _seeds_secs, _m0_seeds)
    _WALL_CLOCK[_label] = _total_secs

    n_seeds = len(SEEDS)
    report = {
        "model": name, "dataset": dataset_name,
        "timestamp": datetime.now().isoformat(),
        "best_params": best_params,
        "val_r2_mean": float(seed_val_r2.mean()), "val_r2_std": float(seed_val_r2.std()),
        "val_r2_ci95": float(1.96 * seed_val_r2.std() / np.sqrt(n_seeds)),
        "val_rmse_mean": float(seed_val_rmse.mean()), "val_rmse_std": float(seed_val_rmse.std()),
        "val_rmse_ci95": float(1.96 * seed_val_rmse.std() / np.sqrt(n_seeds)),
        "val_mae_mean": float(seed_val_mae.mean()), "val_mae_std": float(seed_val_mae.std()),
        "val_mae_ci95": float(1.96 * seed_val_mae.std() / np.sqrt(n_seeds)),
        "val_r2_all_seeds": seed_val_r2.tolist(),
        "val_rmse_all_seeds": seed_val_rmse.tolist(),
        "val_mae_all_seeds": seed_val_mae.tolist(),
        "r2_mean": float(seed_r2.mean()), "r2_std": float(seed_r2.std()),
        "r2_ci95": float(1.96 * seed_r2.std() / np.sqrt(n_seeds)),
        "rmse_mean": float(seed_rmse.mean()), "rmse_std": float(seed_rmse.std()),
        "rmse_ci95": float(1.96 * seed_rmse.std() / np.sqrt(n_seeds)),
        "mae_mean": float(seed_mae.mean()), "mae_std": float(seed_mae.std()),
        "mae_ci95": float(1.96 * seed_mae.std() / np.sqrt(n_seeds)),
        "r2_all_seeds": seed_r2.tolist(),
        "rmse_all_seeds": seed_rmse.tolist(),
        "mae_all_seeds": seed_mae.tolist(),
        "best_seed": int(SEEDS[best_idx]), "best_r2": float(seed_r2.max()),
        "wall_clock_s": round(_total_secs, 1),
        "optuna_s": round(_optuna_secs, 1),
        "seeds_s": round(_seeds_secs, 1),
    }
    with open(os.path.join(SAVE_DIR, f"{tag}_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    print(f"  Saved report to {SAVE_DIR}/{tag}_report.json")
    print(f"  ⏱  Optuna {_optuna_secs:.1f}s + Seeds {_seeds_secs:.1f}s = {_total_secs:.1f}s total")
    return report

## 5.3 Semi TSTR

In [ ]:
REAL_FRACTIONS = [0.05, 0.10, 0.20]

In [ ]:
def objective_semi(trial):
    params = _spec["suggest"](trial)
    model  = _spec["build"](params)
    model.fit(_X_tr, _y_tr)
    return mean_squared_error(_y_va, model.predict(_X_va))

In [ ]:
def run_model_semi_tstr(spec, data_tuple, dataset_name, real_frac=0.10):
    """
    Semi-TSTR:
    1. Split real data into real_train (real_frac) and real_test (rest).
    2. Optuna tunes on synth train/val (seed=42), same as before.
    3. For each seed: combine synth_train + real_train, train, test on real_test.
    4. Report mean ± std across all seeds (val + test).
    """
    global _spec, _X_tr, _y_tr, _X_va, _y_va

    synth_df, real_df = data_tuple
    name = spec["name"]
    TARGET = 'PPP'
    EXCLUDE = ['DEPTH']
    features = [c for c in synth_df.columns if c != TARGET and c not in EXCLUDE]

    print(f"\n{'='*60}")
    print(f"  SEMI-TSTR  {name}  —  {dataset_name}  "
          f"(real_frac={real_frac:.0%}, {len(SEEDS)} seeds)")
    print(f"{'='*60}")

    # ── Step 1: Optuna hyperparameter search (synth only, seed=42) ──────────
    real_sorted = real_df.sort_values("DEPTH").reset_index(drop=True)
    n_real      = len(real_sorted)
    test_cut    = int(0.70 * n_real)
    real_pool   = real_sorted.iloc[:test_cut].reset_index(drop=True)
    real_test   = real_sorted.iloc[test_cut:].reset_index(drop=True)

    rng_hp       = np.random.RandomState(42)
    block_len_hp = max(1, int(real_frac * n_real))
    start_hp     = rng_hp.randint(0, max(1, len(real_pool) - block_len_hp))
    real_tr_hp   = real_pool.iloc[start_hp:start_hp + block_len_hp]
    real_va_hp   = real_pool.drop(real_tr_hp.index)               # disjoint real val

    synth_tr_hp, _ = train_test_split(synth_df, test_size=0.3, random_state=42)
    combo_hp       = pd.concat([synth_tr_hp, real_tr_hp], ignore_index=True)

    scaler_hp     = StandardScaler().fit(combo_hp[features].values)
    tgt_scaler_hp = StandardScaler().fit(combo_hp[[TARGET]].values)

    _X_tr = scaler_hp.transform(combo_hp[features].values)
    _y_tr = tgt_scaler_hp.transform(combo_hp[[TARGET]].values).ravel()
    _X_va = scaler_hp.transform(real_va_hp[features].values)
    _y_va = tgt_scaler_hp.transform(real_va_hp[[TARGET]].values).ravel()
    _spec = spec

    study = optuna.create_study(direction="minimize")
    _t0, _m0 = time.time(), _rss_mb()
    study.optimize(objective_semi, n_trials=N_TRIALS, show_progress_bar=True)
    _optuna_secs = time.time() - _t0
    best_params = study.best_params
    print(f"  Best params: {best_params}")

    # ── Step 2: Multi-seed semi-TSTR evaluation ─────────────────────────────
    seed_r2, seed_rmse, seed_mae, seed_val_r2, seed_val_rmse, seed_val_mae = [], [], [], [], [], []
    _t0_seeds, _m0_seeds = time.time(), _rss_mb()

# ── Fixed, spatially-disjoint real split (once, outside the seed loop) ──
    real_sorted = real_df.sort_values("DEPTH").reset_index(drop=True)
    n_real      = len(real_sorted)
    test_cut    = int(0.70 * n_real)
    real_pool   = real_sorted.iloc[:test_cut].reset_index(drop=True)   # train pool
    real_test   = real_sorted.iloc[test_cut:].reset_index(drop=True)   # FIXED test

    for seed in tqdm(SEEDS, desc=f"  {name} | {dataset_name} | semi {real_frac:.0%}"):
        # Contiguous depth-block sample from real_pool (not random rows)
        rng         = np.random.RandomState(seed)
        block_len   = max(1, int(real_frac * n_real))
        max_start   = max(1, len(real_pool) - block_len)
        start       = rng.randint(0, max_start)
        real_train  = real_pool.iloc[start:start + block_len].reset_index(drop=True)

        synth_train, _ = train_test_split(synth_df, test_size=0.3, random_state=seed)
        combo = pd.concat([synth_train, real_train], ignore_index=True)

        feat_scaler = StandardScaler()
        feat_scaler.fit(combo[features].values)
        tgt_scaler = StandardScaler()
        tgt_scaler.fit(combo[[TARGET]].values)

        X_train = feat_scaler.transform(combo[features].values)
        y_train = tgt_scaler.transform(combo[[TARGET]].values).ravel()
        X_test = feat_scaler.transform(real_test[features].values)

        # Validation = REAL slice disjoint from real_train AND real_test
        real_val  = real_pool.drop(real_train.index, errors="ignore")
        X_val     = feat_scaler.transform(real_val[features].values)

        val_model = spec["build"](best_params)
        val_model.fit(X_train, y_train)
        val_preds = tgt_scaler.inverse_transform(
            val_model.predict(X_val).reshape(-1, 1)
        ).ravel()
        val_truth = real_val[TARGET].values                       # raw real-space

        seed_val_r2.append(r2_score(val_truth, val_preds))
        seed_val_rmse.append(np.sqrt(mean_squared_error(val_truth, val_preds)))
        seed_val_mae.append(mean_absolute_error(val_truth, val_preds))

        model = spec["build"](best_params)
        model.fit(X_train, y_train)
        preds = tgt_scaler.inverse_transform(
            model.predict(X_test).reshape(-1, 1)
        ).ravel()
        truth = real_test[TARGET].values

        seed_r2.append(r2_score(truth, preds))
        seed_rmse.append(np.sqrt(mean_squared_error(truth, preds)))
        seed_mae.append(mean_absolute_error(truth, preds))

    # ── Step 3: Report ──────────────────────────────────────────────────────
    seed_r2       = np.array(seed_r2)
    seed_rmse     = np.array(seed_rmse)
    seed_mae      = np.array(seed_mae)
    seed_val_r2   = np.array(seed_val_r2)
    seed_val_rmse = np.array(seed_val_rmse)
    seed_val_mae  = np.array(seed_val_mae)

    print(f"\n  Results across {len(SEEDS)} seeds (real_frac={real_frac:.0%}):")
    print(f"    Val R²  = {seed_val_r2.mean():.4f} ± {seed_val_r2.std():.4f}   "
          f"(min={seed_val_r2.min():.4f}, max={seed_val_r2.max():.4f})")
    print(f"    Val RMSE= {seed_val_rmse.mean():.4f} ± {seed_val_rmse.std():.4f}")
    print(f"    Val MAE = {seed_val_mae.mean():.4f} ± {seed_val_mae.std():.4f}")
    print(f"    Test R² = {seed_r2.mean():.4f} ± {seed_r2.std():.4f}   "
          f"(min={seed_r2.min():.4f}, max={seed_r2.max():.4f})")
    print(f"    Test RMSE = {seed_rmse.mean():.4f} ± {seed_rmse.std():.4f}")
    print(f"    Test MAE  = {seed_mae.mean():.4f} ± {seed_mae.std():.4f}")
    print(f"    ΔR²     = {(seed_r2.mean() - seed_val_r2.mean()):+.4f}  "
          f"(test - val)")

    if seed_r2.max() < 0:
        print(f"    ⚠️  ALL seeds negative even with {real_frac:.0%} real data.")
    elif seed_r2.mean() < 0:
        pct_pos = 100 * (seed_r2 > 0).sum() / len(seed_r2)
        print(f"    ⚠️  Mean R² negative but {pct_pos:.0f}% of seeds positive.")

    _seeds_secs = time.time() - _t0_seeds
    _total_secs = _optuna_secs + _seeds_secs
    _label = f"{name}__{dataset_name}__semi{int(real_frac*100)}"
    _snap(_label, _t0_seeds - _seeds_secs, _m0_seeds)
    _WALL_CLOCK[_label] = _total_secs

    n_seeds = len(SEEDS)
    report = {
        "mode": "semi-TSTR",
        "model": name, "dataset": dataset_name,
        "real_fraction": real_frac,
        "timestamp": datetime.now().isoformat(),
        "best_params": best_params,
        "val_r2_mean": float(seed_val_r2.mean()), "val_r2_std": float(seed_val_r2.std()),
        "val_r2_ci95": float(1.96 * seed_val_r2.std() / np.sqrt(n_seeds)),
        "val_rmse_mean": float(seed_val_rmse.mean()), "val_rmse_std": float(seed_val_rmse.std()),
        "val_rmse_ci95": float(1.96 * seed_val_rmse.std() / np.sqrt(n_seeds)),
        "val_mae_mean": float(seed_val_mae.mean()), "val_mae_std": float(seed_val_mae.std()),
        "val_mae_ci95": float(1.96 * seed_val_mae.std() / np.sqrt(n_seeds)),
        "val_r2_all_seeds": seed_val_r2.tolist(),
        "val_rmse_all_seeds": seed_val_rmse.tolist(),
        "val_mae_all_seeds": seed_val_mae.tolist(),
        "r2_mean": float(seed_r2.mean()), "r2_std": float(seed_r2.std()),
        "r2_ci95": float(1.96 * seed_r2.std() / np.sqrt(n_seeds)),
        "rmse_mean": float(seed_rmse.mean()), "rmse_std": float(seed_rmse.std()),
        "rmse_ci95": float(1.96 * seed_rmse.std() / np.sqrt(n_seeds)),
        "mae_mean": float(seed_mae.mean()), "mae_std": float(seed_mae.std()),
        "mae_ci95": float(1.96 * seed_mae.std() / np.sqrt(n_seeds)),
        "r2_all_seeds": seed_r2.tolist(),
        "rmse_all_seeds": seed_rmse.tolist(),
        "mae_all_seeds": seed_mae.tolist(),
        "best_seed": int(SEEDS[np.argmax(seed_r2)]),
        "best_r2": float(seed_r2.max()),
        "wall_clock_s": round(_total_secs, 1),
        "optuna_s": round(_optuna_secs, 1),
        "seeds_s": round(_seeds_secs, 1),
    }
    tag = f"semi_{name}_{dataset_name}_frac{int(real_frac*100)}"
    with open(os.path.join(SAVE_DIR, f"{tag}_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    print(f"  Saved → {SAVE_DIR}/{tag}_report.json")
    print(f"  ⏱  Optuna {_optuna_secs:.1f}s + Seeds {_seeds_secs:.1f}s = {_total_secs:.1f}s total")
    return report

## 5.4 Executing per path pure and semi TSTR

In [ ]:
for path_name, data in PATHS_SMALLEST_TO_BIGGEST:
    print(f"\n{'#'*70}")
    print(f"#  PATH: {path_name}")
    print(f"{'#'*70}")

    # Pure TSTR
    for spec in ALL_SPECS:
        report = run_model(spec, data, path_name)
        report["mode"] = "pure-TSTR"
        report["real_fraction"] = 0.0
        all_reports.append(report)

    # Semi-TSTR at 5%, 10%, 20%
    for frac in REAL_FRACTIONS:
        for spec in ALL_SPECS:
            report = run_model_semi_tstr(spec, data, path_name, real_frac=frac)
            all_reports.append(report)

## 5.5 Summary table

In [ ]:
print(f"\n{'='*80}")
print(f"  FULL TSTR SUMMARY  (Pure + Semi)")
print(f"{'='*80}")
print(f"  {'Well':<10} {'Model':<14} {'Mode':<12} {'Frac':<6} {'R² mean':>10} {'± std':>10} {'RMSE mean':>12}")
print(f"  {'-'*74}")
for r in all_reports:
    frac = r.get('real_fraction', 0.0)
    mode = r.get('mode', 'pure-TSTR' if frac == 0.0 else 'semi-TSTR')
    frac_str = "0%" if frac == 0.0 else f"{frac:.0%}"
    print(f"  {r['dataset']:<10} {r['model']:<14} {mode:<12} {frac_str:<6} "
          f"{r['r2_mean']:>10.4f} {r['r2_std']:>10.4f} {r['rmse_mean']:>12.4f}")

In [ ]:
rows = []
for r in all_reports:
    rows.append({
        "well":       r["dataset"],
        "model":      r["model"],
        "mode":       r.get("mode", "pure-TSTR"),
        "real_frac":  r.get("real_fraction", 0.0),
        "r2_mean":    r["r2_mean"],
        "r2_std":     r["r2_std"],
        "r2_ci95":    r.get("r2_ci95", 1.96 * r["r2_std"] / np.sqrt(len(SEEDS))),
        "rmse_mean":  r["rmse_mean"],
        "rmse_std":   r["rmse_std"],
        "rmse_ci95":  r.get("rmse_ci95", 1.96 * r["rmse_std"] / np.sqrt(len(SEEDS))),
        "mae_mean":   r["mae_mean"],
        "mae_std":    r["mae_std"],
        "mae_ci95":   r.get("mae_ci95", 1.96 * r["mae_std"] / np.sqrt(len(SEEDS))),
        "wall_clock_s": r.get("wall_clock_s", None),
    })

table3 = pd.DataFrame(rows)
table3.to_csv(os.path.join(SAVE_DIR, "table3_tstr_all.csv"), index=False)
print(f"\n>>> Saved {SAVE_DIR}/table3_tstr_all.csv  ({len(table3)} rows)")
print(table3.to_string(index=False, float_format="%.4f"))

In [ ]:
_total_all = time.time() - _t0_all
print(f"\n{'='*70}")
print("COMPUTE COST REPORT")
print(f"{'='*70}")
print(f"  {'Phase':<45s} {'Wall (s)':>10} {'(min)':>7} {'ΔRSS (MB)':>10}")
print(f"  {'-'*72}")
for phase in _WALL_CLOCK:
    s = _WALL_CLOCK[phase]
    m = _PEAK_MEM.get(phase, 0)
    print(f"  {phase:<45s} {s:>10.1f} {s/60:>7.1f} {m:>10.1f}")
print(f"  {'-'*72}")
print(f"  {'GRAND TOTAL':<45s} {_total_all:>10.1f} {_total_all/60:>7.1f} {_rss_mb():>10.1f}")

if torch.cuda.is_available():
    print(f"\n  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU peak mem allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    print(f"  GPU peak mem reserved:  {torch.cuda.max_memory_reserved()/1e9:.2f} GB")
else:
    print(f"\n  GPU: not used (CPU only)")